# Counterfactual Explanations for Classification with CELIA

**What are counterfactual explanations?**

Counterfactual explanations answer the question: *"What would need to change about this input for the model to produce a different outcome?"*

Imagine Alice applies for a loan and gets denied. A counterfactual explanation might say:
*"If your annual income were $\$62,000$ instead of $\$45,000$ and your credit score were 710 instead of 650, you would have been approved."*

In this tutorial we will:
1. Create a synthetic loan approval dataset
2. Train a Random Forest classifier
3. Define real-world constraints (immutable features, feasible ranges)
4. Wrap the data and model for CELIA
5. Select a denied applicant and inspect the model's prediction
6. Generate counterfactual explanations with three methods: **DiCE**, **Growing Spheres**, and **NNCE**
7. Compare the results side-by-side

In [1]:
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

from celia.data import Data
from celia.model import SklearnModel
from celia.explainers import DiceClassifierExplainer, GrowingSpheresClassifierExplainer, NNCEClassifierExplainer

## 1. Creating the Synthetic Loan Approval Dataset

We generate a synthetic dataset where a loan approval decision depends on six numeric features.
All features are kept numeric so that every explainer method works out of the box — Growing Spheres (GSG) requires all features to be numeric.

| Feature | Description | Range |
|---------|-------------|-------|
| `age` | Applicant's age | 21 – 65 |
| `annual_income` | Yearly income in thousands (\$) | 20 – 150 |
| `credit_score` | Credit score | 300 – 850 |
| `employment_years` | Years at current employer | 0 – 40 |
| `debt_to_income_ratio` | Monthly debt / monthly income | 0.0 – 1.0 |
| `num_credit_lines` | Number of open credit lines | 0 – 20 |
| **`approved`** | **Target: 0 = Denied, 1 = Approved** | **0 or 1** |

In [2]:
def create_loan_dataset(n_samples=1000, random_state=42):
    """Generate a synthetic loan-approval dataset.

    Approval is based on a weighted score with noise, producing a
    realistic (non-perfectly-separable) binary classification problem.
    """
    rng = np.random.default_rng(random_state)

    age = rng.integers(21, 66, size=n_samples)
    annual_income = rng.uniform(20, 150, size=n_samples).round(1)
    credit_score = rng.integers(300, 851, size=n_samples)
    employment_years = rng.integers(0, 41, size=n_samples)
    debt_to_income_ratio = rng.uniform(0.0, 1.0, size=n_samples).round(3)
    num_credit_lines = rng.integers(0, 21, size=n_samples)

    # Higher score → more likely to be approved
    score = (
        0.01 * (credit_score - 300)
        + 0.005 * annual_income
        + 0.02 * employment_years
        - 1.5 * debt_to_income_ratio
        + 0.01 * (age - 21)
        + 0.05 * num_credit_lines
        + rng.normal(0, 0.5, size=n_samples)
    )
    approved = (score > 3.5).astype(int)

    X = pd.DataFrame({
        "age": age,
        "annual_income": annual_income,
        "credit_score": credit_score,
        "employment_years": employment_years,
        "debt_to_income_ratio": debt_to_income_ratio,
        "num_credit_lines": num_credit_lines,
    })
    y = pd.Series(approved, name="approved")

    return X, y

In [3]:
X, y = create_loan_dataset(n_samples=1000, random_state=42)

print(f"Dataset shape: {X.shape}")
print(f"Approval rate: {y.mean():.1%}\n")
print(f"Class distribution:\n{y.value_counts().to_string()}\n")
X.describe().round(2)

Dataset shape: (1000, 6)
Approval rate: 49.9%

Class distribution:
approved
0    501
1    499



,age,annual_income,credit_score,employment_years,debt_to_income_ratio,num_credit_lines
count,1000.00,1000.00,1000.00,1000.00,1000.00,1000.00
mean,43.14,85.21,575.62,19.54,0.50,9.85
std,13.00,38.21,157.60,11.98,0.29,5.99
min,21.00,20.10,300.00,0.00,0.00,0.00
25%,32.00,51.70,433.00,10.00,0.24,5.00
50%,43.00,84.15,575.00,19.00,0.51,10.00
75%,55.00,119.12,712.00,30.00,0.75,15.00
max,65.00,149.90,850.00,40.00,1.00,20.00


## 2. Training a Classification Model

We train a `RandomForestClassifier` — a strong baseline that also provides `predict_proba`, which is required by the Growing Spheres method.

In [4]:
def train_model(X, y, test_size=0.2, random_state=42):
    """Split data, train a RandomForestClassifier, and print metrics."""
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, stratify=y, random_state=random_state
    )

    model = RandomForestClassifier(n_estimators=200, max_depth=8, random_state=random_state)
    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)
    print(f"Accuracy: {accuracy_score(y_test, y_pred):.2%}\n")
    print(classification_report(y_test, y_pred, target_names=["Denied", "Approved"]))

    return model, X_train, X_test, y_train, y_test

In [5]:
model, X_train, X_test, y_train, y_test = train_model(X, y)

Accuracy: 94.50%

              precision    recall  f1-score   support

      Denied       0.95      0.94      0.94       100
    Approved       0.94      0.95      0.95       100

    accuracy                           0.94       200
   macro avg       0.95      0.94      0.94       200
weighted avg       0.95      0.94      0.94       200



## 3. Defining Real-World Constraints

Before generating counterfactual explanations, we define what changes are **realistic**:

- **Immutable features:** A person's `age` cannot be changed by a loan officer's advice — we mark it as immutable.
- **Feasible value ranges:** Features have natural boundaries. A credit score cannot exceed 850, income cannot be negative, etc.

These constraints ensure the suggested changes are actionable and meaningful.

In [6]:
# Features that cannot change in a counterfactual
immutable_features = ["age"]

# All features are continuous (numeric) in this dataset
continuous_features = [
    "age", "annual_income", "credit_score",
    "employment_years", "debt_to_income_ratio", "num_credit_lines",
]

# Realistic bounds for each feature
feasible_values = {
    "age": (21, 65),
    "annual_income": (20.0, 150.0),
    "credit_score": (300, 850),
    "employment_years": (0, 40),
    "debt_to_income_ratio": (0.0, 1.0),
    "num_credit_lines": (0, 20),
}

## 4. Wrapping Data and Model for CELIA

CELIA uses two wrapper objects:
- **`Data`** — wraps the training data along with metadata (feature types, constraints, feasible values). Every explainer reads constraints from this object.
- **`SklearnModel`** — wraps a scikit-learn model to provide a unified prediction interface.

In [7]:
public_data = Data(
    data=X_train,
    targets=y_train,
    target_name="approved",
    continuous_column_names=continuous_features,
    immutable_column_names=immutable_features,
    feasible_values=feasible_values,
)

sklearn_model = SklearnModel(model)

print(f"Data created with {len(X_train)} training samples")
print(f"Continuous features:  {public_data.continuous_column_names}")
print(f"Immutable features:   {public_data.immutable_column_names}")
print(f"Feasible value ranges: {public_data.feasible_values}")

Data created with 800 training samples
Continuous features:  ['age', 'annual_income', 'credit_score', 'employment_years', 'debt_to_income_ratio', 'num_credit_lines']
Immutable features:   ['age']
Feasible value ranges: {'age': (21, 65), 'annual_income': (20.0, 150.0), 'credit_score': (300, 850), 'employment_years': (0, 40), 'debt_to_income_ratio': (0.0, 1.0), 'num_credit_lines': (0, 20)}


## 5. Meet Alice: A Denied Loan Applicant

Let's find a test instance where the model predicts **Denied** (class 0). This is our "Alice" — a loan applicant who wants to understand what she could change to get approved.

In [8]:
# Find test instances predicted as "Denied" (class 0)
test_predictions = model.predict(X_test)
denied_indices = X_test.index[test_predictions == 0]

# Pick the first denied applicant as "Alice"
alice = X_test.loc[[denied_indices[0]]]

print("Alice's loan application:")
display(alice)
print(f"\nModel prediction:     {'Approved' if model.predict(alice)[0] == 1 else 'Denied'}")
print(f"Approval probability: {model.predict_proba(alice)[0][1]:.1%}")

Alice's loan application:


,age,annual_income,credit_score,employment_years,debt_to_income_ratio,num_credit_lines
585,62,116.8,419,25,0.316,0



Model prediction:     Denied
Approval probability: 8.9%


## 6. Generating Counterfactual Explanations

Now we ask three different methods: *"What would Alice need to change to get approved?"*

Each method approaches this question differently:

| Method | Approach | Strengths |
|--------|----------|-----------|
| **DiCE** | Optimization-based; generates diverse CFs | Multiple diverse suggestions; respects constraints |
| **Growing Spheres (GSG)** | Geometric search around the instance | Finds the closest decision boundary crossing |
| **NNCE** | Nearest-neighbor search in training data | Returns real training examples; always feasible |

### 6.1 DiCE — Diverse Counterfactual Explanations

DiCE generates multiple, diverse counterfactuals by solving an optimization problem. It balances **proximity** to the original instance with **diversity** among the suggestions.

- `total_CFs=3` asks for three alternative paths to approval.
- Immutable features and feasible ranges are read automatically from `Data`.
- `method="random"` uses random sampling to find candidates.

In [9]:
dice_explainer = DiceClassifierExplainer(
    model=sklearn_model,
    data=public_data,
    method="random",
)

# DiCE returns a list (one Counterfactual per input row)
dice_results = dice_explainer.generate_counterfactuals(alice, total_CFs=3)
dice_cf = dice_results[0]

print("DiCE: What changes would get Alice approved?\n")
dice_cf.highlighted_counterfactuals

DiCE: What changes would get Alice approved?



,age,annual_income,credit_score,employment_years,debt_to_income_ratio,num_credit_lines,Prediction
0,-,116.8,737,-,0.316,18,0.0 → 1
1,-,116.8,689,31,0.316,-,0.0 → 1
2,-,116.8,619,-,0.316,-,0.0 → 1


The `highlighted_counterfactuals` view shows only the values that **differ** from Alice's original application — unchanged values appear as `"-"`. Notice how `age` is always `"-"` because we marked it as immutable.

Each row is a different suggestion: DiCE deliberately generates **diverse** alternatives so Alice can see multiple paths to approval.

### 6.2 Growing Spheres (GSG) — Geometric Search

Growing Spheres works by expanding a hyperspherical shell around Alice's profile until it finds a candidate that crosses the decision boundary. It then shrinks back to find the **closest** point where the prediction flips.

- Produces exactly **1 counterfactual** — the nearest boundary crossing.
- Requires `predict_proba` (which our RandomForest provides).
- Requires all-numeric features (which our dataset satisfies).

In [10]:
gsg_explainer = GrowingSpheresClassifierExplainer(
    model=sklearn_model,
    data=public_data,
    n_samples=1000,
    seed=42,
)

# GSG returns a single Counterfactual for a single-row input
gsg_cf = gsg_explainer.generate_counterfactuals(alice)

print("Growing Spheres: The smallest change to get Alice approved\n")
gsg_cf.highlighted_counterfactuals

Growing Spheres: The smallest change to get Alice approved



,age,annual_income,credit_score,employment_years,debt_to_income_ratio,num_credit_lines,Prediction
585,-,114.075477,444.316414,43.576366,-1.899009,15.335342,0 → 1


Notice how Growing Spheres provided a counterfactual where ``debt_to_income_ratio`` has a negative value. This shows a weakness in this particularly method, that does not enforce feasible values ranges. This highlights how not every method is fit for every task.

### 6.3 NNCE — Nearest Neighbor Counterfactual Explanations

NNCE takes a fundamentally different approach: instead of generating synthetic counterfactuals, it searches the **actual training data** for the nearest instances with a different prediction.

- Returns **real data points** (not synthetic), so they are always realistic.
- `n_counterfactuals=3` returns the three closest approved applicants.
- Respects immutable features (only considers neighbors matching Alice on immutable columns).

In [11]:
nnce_explainer = NNCEClassifierExplainer(
    model=sklearn_model,
    data=public_data,
)

# NNCE returns a single Counterfactual (with multiple rows) for a single-row input
nnce_cf = nnce_explainer.generate_counterfactuals(alice, n_counterfactuals=3)

print("NNCE: Real approved applicants most similar to Alice\n")
nnce_cf.highlighted_counterfactuals

NNCE: Real approved applicants most similar to Alice



,age,annual_income,credit_score,employment_years,debt_to_income_ratio,num_credit_lines,Prediction
0,-,32.6,584,29,0.999,15,0 → 1
1,-,113.8,624,1,0.310,-,0 → 1
2,-,106.3,652,10,0.956,19,0 → 1


These are **real people** from the training data who got approved and are most similar to Alice. This makes NNCE results particularly convincing — they are existence proofs, not hypothetical scenarios.

## 7. Comparing the Three Methods

Let's put Alice's original profile side-by-side with the highlighted changes from each method.

In [12]:
print("=" * 80)
print("ALICE'S ORIGINAL APPLICATION")
print("=" * 80)
display(alice)
print(f"Model prediction: {'Approved' if dice_cf.original_prediction == 1 else 'Denied'}")
print(f"Approval probability: {model.predict_proba(alice)[0][1]:.1%}")

print("\n" + "=" * 80)
print("DiCE — 3 diverse suggestions for approval")
print("=" * 80)
display(dice_cf.highlighted_counterfactuals)

print("\n" + "=" * 80)
print("Growing Spheres — smallest change to cross the decision boundary")
print("=" * 80)
display(gsg_cf.highlighted_counterfactuals)

print("\n" + "=" * 80)
print("NNCE — real approved applicants most similar to Alice")
print("=" * 80)
display(nnce_cf.highlighted_counterfactuals)

ALICE'S ORIGINAL APPLICATION


,age,annual_income,credit_score,employment_years,debt_to_income_ratio,num_credit_lines
585,62,116.8,419,25,0.316,0


Model prediction: Denied
Approval probability: 8.9%

DiCE — 3 diverse suggestions for approval


,age,annual_income,credit_score,employment_years,debt_to_income_ratio,num_credit_lines,Prediction
0,-,116.8,737,-,0.316,18,0.0 → 1
1,-,116.8,689,31,0.316,-,0.0 → 1
2,-,116.8,619,-,0.316,-,0.0 → 1



Growing Spheres — smallest change to cross the decision boundary


,age,annual_income,credit_score,employment_years,debt_to_income_ratio,num_credit_lines,Prediction
585,-,114.075477,444.316414,43.576366,-1.899009,15.335342,0 → 1



NNCE — real approved applicants most similar to Alice


,age,annual_income,credit_score,employment_years,debt_to_income_ratio,num_credit_lines,Prediction
0,-,32.6,584,29,0.999,15,0 → 1
1,-,113.8,624,1,0.310,-,0 → 1
2,-,106.3,652,10,0.956,19,0 → 1


## Key Takeaways

**What Alice learned:**
- All three methods suggest a general direction: to get approved, Alice likely needs some combination of higher credit score, higher income, or lower debt-to-income ratio.
- Her `age` was never modified — the immutable constraint was respected by all methods.
- DiCE and NNCE suggested values fall within the feasible ranges we defined. Growing Spheres unfortunately did not respect the feasible range for `debt_to_income_ratio`, which is a known limitation of that method.

**Important caveat:** Counterfactual explanations describe the *model's* behavior, not necessarily causal relationships. "Increase your credit score to 720" tells you what the model would predict, not that doing so guarantees real-world approval.